# 01 - Frame

*Question:* At pickup, predict whether each trip will crawl through traffic, so that the dispatcher can warn the passenger.


In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, precision_score, recall_score
from sklearn.tree import DecisionTreeClassifier, export_text

RANDOM_STATE = 42

trips = pd.read_parquet("../data/raw/yellow_tripdata_2026_sample.parquet")

# Data
One row is one taxi trip. There are 200,000 trips from each of May, June and July 2026, raw.

In [2]:
print(trips.shape)
trips.head()

(600000, 21)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,request_source
0,2,2026-04-18 21:51:07,2026-04-18 22:09:09,1.0,3.06,1.0,N,263,170,1,...,1.0,0.5,3.62,0.0,1.0,27.77,2.5,0.0,0.75,NaN
1,2,2026-04-30 23:51:53,2026-04-30 23:58:30,1.0,1.06,1.0,N,170,137,1,...,1.0,0.5,0.05,0.0,1.0,14.40,2.5,0.0,0.75,NaN
2,2,2026-05-01 00:00:28,2026-05-01 00:05:00,1.0,0.89,1.0,N,151,238,1,...,1.0,0.5,1.00,0.0,1.0,13.20,2.5,0.0,0.00,NaN
3,2,2026-05-01 00:00:34,2026-05-01 00:22:40,1.0,3.45,1.0,N,144,80,1,...,1.0,0.5,8.93,0.0,1.0,38.68,2.5,0.0,0.75,NaN
4,2,2026-05-01 00:00:51,2026-05-01 00:31:10,1.0,12.00,1.0,N,138,49,1,...,6.0,0.5,11.62,0.0,1.0,71.72,0.0,2.0,0.00,NaN


# Build the target
A trip crawls when its average speed is below 6 mph. Average speed is distance divided by duration, and those columns don't exists, so we build them.

In [3]:
trips["duration_min"] = (trips["tpep_dropoff_datetime"] - trips["tpep_pickup_datetime"]).dt.total_seconds() / 60
trips["duration_min"].describe()

count    600000.000000
mean         17.853690
std          23.872686
min           0.000000
25%           8.500000
50%          14.233333
75%          22.550000
max        4253.850000
Name: duration_min, dtype: float64

The extremes are not acceptable as data, the shortest trips last 0 minutes and the longest about 71 hours. We will deal woth that afterwards.


The trip_distance column is in miles, so the duration must be in hours for the result to be in mph. The is_crawl column is 1 when the speed is below 6 mph and 0 otherwise.

These columns use the drop-off time, which only exists once the trip has ended. They build the answer, so they can never be used as inputs to the model.

In [4]:
trips["speed_mph"] = trips["trip_distance"] / (trips["duration_min"] / 60)
trips["is_crawl"] = (trips["speed_mph"] < 6).astype(int)

# Trip filtering

Each rule is a fixed threshold, not a statistic learned from the data, so applying it before the split cannot leak information between months.

| Rule | Why |
|---|---|
| Duration ≥ 1 minute | Shorter "trips" are meters switched on and off, or voided trips. A duration of 0 also makes the speed infinite. |
| Duration ≤ 120 minutes | Above two hours, durations are dominated by meters left running (the longest was about 71 hours). These errors give very low speeds, so they would be labelled as crawls that never happened. |
| Distance > 0 miles | A zero distance gives a speed of 0, which would be labelled as a crawl even though the taxi never moved. |
| Distance ≤ 40 miles | Some distances are absurd (the longest was 317,017 miles). Real trips above 40 miles are very rare. |
| Speed < 70 mph | Some trips pass every rule above, yet the combination is impossible (up to 534 mph). No taxi trip in New York is that fast. |

In [5]:
trips["valid"] = (
    (trips["duration_min"] >= 1)
    & (trips["duration_min"] <= 120)
    & (trips["trip_distance"] > 0)
    & (trips["trip_distance"] <= 40)
    & (trips["speed_mph"] < 70)
)

print("valid trips:  ", trips["valid"].sum())
print("removed trips:", (~trips["valid"]).sum())

valid trips:   570327
removed trips: 29673


Together the rules remove 29,673 of 600,000 trips (5%), and the positive rate settles at 18.5%, about one trip in five.

The speed of the trips we keep looks plausible: half of them run between about 7 and 13 mph.

In [6]:
trips.loc[trips["valid"], "speed_mph"].describe()

count    570327.000000
mean         10.715788
std           6.294930
min           0.005803
25%           6.713115
50%           9.201544
75%          12.838235
max          69.321663
Name: speed_mph, dtype: float64

# Flex fare trips

Flex fare trips have the payment_type set to 0. They are about a quarter of the valid trips, and their passenger_count is set to 0.

In [7]:
flex = trips["payment_type"] == 0
valid = trips["valid"]

summary = pd.DataFrame({
    "share_of_valid_trips": [(valid & flex).sum() / valid.sum(), (valid & ~flex).sum() / valid.sum()],
    "positive_rate": [trips.loc[valid & flex, "is_crawl"].mean(), trips.loc[valid & ~flex, "is_crawl"].mean()],
    "median_distance_miles": [trips.loc[valid & flex, "trip_distance"].median(), trips.loc[valid & ~flex, "trip_distance"].median()],
}, index=["flex", "not flex"])
summary.round(3)

,share_of_valid_trips,positive_rate,median_distance_miles
flex,0.247,0.115,3.0
not flex,0.753,0.208,1.7


Flex trips crawl much less often (11.5% to 20.8%), mostly because they are longer, and longer trips use faster roads usually.

*Decision: we keep them.*
The dispatcher would face these trips in practice, so dropping them would make the model look better than it would be in use.

Two consequences:
- The passenger_count is missing for them, so the pipeline must fill it in or leave it out.
- In the evaluation, results should be checked for flex and non-flex trips separately.

# The data split

May -> Training /
June -> Testing /
July -> Validating

We ignore the trips from other months.

In [8]:
labelled = trips[trips["valid"]].copy()
month = labelled["tpep_pickup_datetime"].dt.month
labelled["split"] = month.map({5: "train", 6: "valid", 7: "test"})

print("trips outside May to July:", labelled["split"].isna().sum())
labelled = labelled[labelled["split"].notna()]

train = labelled[labelled["split"] == "train"]
valid = labelled[labelled["split"] == "valid"]

print("train:", train.shape, "positive rate", round(train["is_crawl"].mean(), 3))
print("valid:", valid.shape, "positive rate", round(valid["is_crawl"].mean(), 3))

trips outside May to July: 2
train: (190878, 26) positive rate 0.197
valid: (190003, 26) positive rate 0.19


# The baselines

Before any serious model, three rungs, each one the bar for the next. Each is designed on May and scored on June.

The metrics are those in PROBLEM.md: 
precision and recall at the threshold, PR AUC across all thresholds, and lift = PR AUC ÷ positive rate (1 means no better than guessing).
Accuracy is not used: a model that never warns would score about 81%.

In [9]:
results = []


def evaluate(name, y_true, y_pred, y_score):
    """Score one rung on June and add it to the results table."""
    pr_auc = average_precision_score(y_true, y_score)
    results.append({
        "model": name,
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred),
        "pr_auc": pr_auc,
        "lift": pr_auc / y_true.mean(),
    })
    return pd.DataFrame(results).round(3)


y_valid = valid["is_crawl"]

### 1 - No skill

Always predict "does not crawl". It never sends a warning, so its precision and recall are 0, and its lift is 1 by definition.

In [10]:
zeros = np.zeros(len(valid), dtype=int)
evaluate("1. always no crawl", y_valid, zeros, zeros)

,model,precision,recall,pr_auc,lift
0,1. always no crawl,0.0,0.0,0.19,1.0


### Rung 2 - Rule a dispatcher could write

To write the rule, we look at when and where trips crawl (only May)

In [11]:
pickup = train["tpep_pickup_datetime"]
by_hour = train.groupby(pickup.dt.hour)["is_crawl"].mean().round(3)
by_hour

tpep_pickup_datetime
0     0.057
1     0.044
2     0.027
3     0.018
4     0.013
5     0.011
6     0.020
7     0.066
8     0.140
9     0.199
10    0.250
11    0.288
12    0.309
13    0.296
14    0.300
15    0.317
16    0.298
17    0.286
18    0.244
19    0.200
20    0.127
21    0.103
22    0.092
23    0.070
Name: is_crawl, dtype: float64

Crawling is a daytime problem: below 7% at night, and around 25–32% between 10:00 and 18:59.

Next, the pickup zones with the highest crawl rate. Zones with fewer than 500 trips in May are left out to avoid noise.

In [12]:
by_zone = train.groupby("PULocationID")["is_crawl"].agg(["size", "mean"])
by_zone = by_zone[by_zone["size"] >= 500].sort_values("mean", ascending=False)
by_zone.head(10).round(3)

,size,mean
PULocationID,,
161,7746,0.383
186,5410,0.352
163,4627,0.331
162,5693,0.328
229,3079,0.322
100,2380,0.321
230,5246,0.310
170,4878,0.306
164,3501,0.294


The top zones are all in Midtown Manhattan (161 Midtown Center, 186 Penn Station, 163 Midtown North, etc.).

*The rule:* warn when the pickup is between 10:00 and 18:59 in one of the 10 zones with the highest crawl rate in May.

In [13]:
RULE_HOURS = range(10, 19)
RULE_ZONES = by_zone.head(10).index.tolist()


def rule_predict(df):
    hour = df["tpep_pickup_datetime"].dt.hour
    return (hour.isin(RULE_HOURS) & df["PULocationID"].isin(RULE_ZONES)).astype(int)


pred_rule = rule_predict(valid)
evaluate("2. rule: 10-18h in top 10 zones", y_valid, pred_rule, pred_rule)

,model,precision,recall,pr_auc,lift
0,1. always no crawl,0.000,0.000,0.190,1.000
1,2. rule: 10-18h in top 10 zones,0.445,0.349,0.279,1.468


### 3 - A tiny model

A decision tree two levels deep, trained on May. The inputs are allowed at pickup: the hour, the day of the week and trip_distance (planned route).

A trip is flagged when its predicted probability is above May's positive rate. This threshold is a placeholder, the real one is chosen from the costs of each mistake in the evaluation.

In [14]:
def make_X(df):
    return pd.DataFrame({
        "hour": df["tpep_pickup_datetime"].dt.hour,
        "day_of_week": df["tpep_pickup_datetime"].dt.day_of_week,
        "trip_distance": df["trip_distance"],
    })


X_train, y_train = make_X(train), train["is_crawl"]
X_valid = make_X(valid)

tree = DecisionTreeClassifier(max_depth=2, random_state=RANDOM_STATE)
tree.fit(X_train, y_train)
print(export_text(tree, feature_names=list(X_train.columns)))

score_tree = tree.predict_proba(X_valid)[:, 1]
pred_tree = (score_tree >= y_train.mean()).astype(int)

evaluate("3. tree, depth 2", y_valid, pred_tree, score_tree)

|--- trip_distance <= 1.92
|   |--- trip_distance <= 0.33
|   |   |--- class: 1
|   |--- trip_distance >  0.33
|   |   |--- class: 0
|--- trip_distance >  1.92
|   |--- trip_distance <= 3.41
|   |   |--- class: 0
|   |--- trip_distance >  3.41
|   |   |--- class: 0



,model,precision,recall,pr_auc,lift
0,1. always no crawl,0.000,0.000,0.190,1.000
1,2. rule: 10-18h in top 10 zones,0.445,0.349,0.279,1.468
2,"3. tree, depth 2",0.315,0.833,0.339,1.781


The bar to beat is the depth-2 tree: PR AUC 0.339, lift 1.78 on June. Every model from now on is compared against it.